In [0]:
# ══════════════════════════════════════
# 00_CONFIG — Configuracoes do Projeto
# Squad 3 — Batch Ecommerce
# ══════════════════════════════════════

# Instalacao

%pip install python-dotenv azure-storage-file-datalake azure-identity

In [0]:
# Imports

import os
import pandas as pd
from io import BytesIO
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

load_dotenv(override=True)
print("Imports carregados!")

In [0]:
# Configurar Acesso ao ADLS

def configurar_acesso_adls():
    storage_account   = os.getenv("storage_account_name")
    client_id_val     = os.getenv("client_id")
    tenant_id_val     = os.getenv("tenant_id")
    client_secret_val = os.getenv("client_secret")

    if not all([storage_account, client_id_val, tenant_id_val, client_secret_val]):
        raise ValueError("Uma ou mais variaveis do .env estao vazias.")

    credential = ClientSecretCredential(
        tenant_id=tenant_id_val,
        client_id=client_id_val,
        client_secret=client_secret_val
    )

    account_url = f"https://{storage_account}.dfs.core.windows.net"
    client      = DataLakeServiceClient(
        account_url=account_url,
        credential=credential
    )

    print(f"Acesso ao ADLS configurado para: {storage_account}")
    return client, storage_account

adls_client, storage_account = configurar_acesso_adls()
container = os.getenv("container_name")
print(f"Container ativo: {container}")

In [0]:
# Configuracao SQL Server

def get_sql_options():
    host     = os.getenv("jdbc_hostname")
    database = os.getenv("jdbc_database")
    username = os.getenv("jdbc_username")
    password = os.getenv("jdbc_password")

    if not all([host, database, username, password]):
        raise ValueError("Uma ou mais variaveis SQL do .env estao vazias.")

    options = {
        "host"                   : host,
        "port"                   : "1433",
        "database"               : database,
        "user"                   : username,
        "password"               : password,
        "encrypt"                : "true",
        "trustServerCertificate" : "false"
    }

    print("Configuracao SQL Server criada!")
    return options

sql_options = get_sql_options()


In [0]:
# Explorar Estrutura do Container

def listar_arquivos_path(adls_client, container, path=""):
    fs_client = adls_client.get_file_system_client(file_system=container)
    arquivos  = [p.name for p in fs_client.get_paths(path=path) if not p.is_directory]

    print(f"Arquivos em '{path}':")
    for arquivo in arquivos:
        print(f"   - {arquivo}")

    return arquivos

print("\n=== Raiz do container ===")
listar_arquivos_path(adls_client, container, path="")

print("\n=== Dentro de batch-data ===")
listar_arquivos_path(adls_client, container, path="batch-data")

print("\n00_config carregado com sucesso!")